## Practice Lab 22- Recursive Neural Networks
In this lab we will look at Recursive Neural Networks. \
Based on Chapter 15 from Aurelien Geron's book, Hands-on Machine Learning with Scikit-Learn Keras & Tensorflow.\
Original code examples from book in github [here](https://github.com/ageron/handson-ml2)


<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/dtrad/geoml_course/blob/master/Practice22_RNN.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

try:
    # %tensorflow_version only exists in Colab.
    %tensorflow_version 2.x
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. LSTMs and CNNs can be very slow without a GPU.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")

# Add for GPU BEFORE JSON
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession

config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)
####################################

        
# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt

## Exercise 1 - Predictions of time series using different methods
Using the time series generator below, predict future samples by using:
1) Linear regression\
2) Dense Neural network\
3) Recursive Neural network\
4) Deep Recursive Neural network\


In [ ]:
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))  #   wave 1
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20)) # + wave 2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)   # + noise
    return series[..., np.newaxis].astype(np.float32)

Let us generate 7000 time series, each with 50 samples and an extra sample to predict (later we will predict more than one) 

In [ ]:
np.random.seed(42)
n_steps = 50
series = generate_time_series(10000, n_steps + 1)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

In [ ]:
print(X_train.shape, X_valid.shape, X_test.shape)
print(y_train.shape, y_valid.shape, y_test.shape)

In [ ]:
print(y_train.shape, y_valid.shape, y_test.shape)

In [ ]:
ts=[0,1,10,23]
plt.plot(X_train[ts[0],:,:],'r-')
plt.plot(X_train[ts[1],:,:],'g-')
plt.plot(X_train[ts[2],:,:],'b-')
plt.plot(X_train[ts[3],:,:],'y-')



In [ ]:
def plot_series(series, y=None, y_pred=None, x_label="$t$", y_label="$x(t)$"):
    plt.plot(series, ".-")
    if y is not None:
        plt.plot(n_steps, y, "bx", markersize=10)
    if y_pred is not None:
        plt.plot(n_steps, y_pred, "ro")
    plt.grid(True)
    if x_label:
        plt.xlabel(x_label, fontsize=16)
    if y_label:
        plt.ylabel(y_label, fontsize=16, rotation=0)
    plt.hlines(0, 0, 100, linewidth=1)
    plt.axis([0, n_steps + 1, -1, 1])

fig, axes = plt.subplots(nrows=1, ncols=3, sharey=True, figsize=(12, 4))
for col in range(3):
    plt.sca(axes[col])
    plot_series(X_valid[col, :, 0], y_valid[col, 0],
                y_label=("$x(t)$" if col==0 else None))

plt.show()

The simplest prediction we can do is to repeat the last sample

In [ ]:
y_pred_naive=X_valid[:,-1]
np.mean(keras.losses.mean_squared_error(y_valid, y_pred_naive))

In [ ]:
plot_series(X_valid[0, :, 0], y_valid[0, 0], y_pred_naive[0, 0])

Now let us make a linear prediction using a dense layer

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50, 1]),
    keras.layers.Dense(1)
])

model.compile(loss="mse", optimizer="adam")
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))

Let us compare with the naive approach (copying the last sample)

In [ ]:
print('MSE naive prediction',np.mean(keras.losses.mean_squared_error(y_valid, X_valid[:,-1])))
print('MSE error linear model',model.evaluate(X_valid, y_valid))

In [ ]:
y_pred_DN = model.predict(X_valid)
plot_series(X_valid[0, :, 0], y_valid[0, 0], y_pred_DN[0, 0])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(12, 4))
col=0
plt.subplot(121),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_naive[col, 0])
plt.subplot(122),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_DN[col, 0])

Now let us use a recursive neural network. Let start with the simplest possible,one layer and one neuron.

In [ ]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

optimizer = keras.optimizers.Adam(lr=0.005)
model.compile(loss="mse", optimizer=optimizer)
history = model.fit(X_train, y_train, epochs=20,validation_data=(X_valid, y_valid))

That took much longer than the DN. Let us evaluate it:

In [ ]:
print('MSE naive prediction',np.mean(keras.losses.mean_squared_error(y_valid, X_valid[:,-1])))
print('MSE error RNN model',model.evaluate(X_valid, y_valid))

In [ ]:
y_pred_RNN = model.predict(X_valid)
plot_series(X_valid[0, :, 0], y_valid[0, 0], y_pred_RNN[0, 0])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, sharey=True, figsize=(12, 4))
col=1
plt.subplot(131),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_naive[col, 0])
plt.subplot(132),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_DN[col, 0])
plt.subplot(133),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_RNN[col, 0])


The DN (linear) prediction is better than the RNN. Let us try more layers.

In [ ]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.SimpleRNN(1)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, y_train, epochs=20,validation_data=(X_valid, y_valid))

In [ ]:
print('MSE naive prediction',np.mean(keras.losses.mean_squared_error(y_valid, X_valid[:,-1])))
print('MSE error RNN model',model.evaluate(X_valid, y_valid))

In [ ]:
y_pred_DRNN = model.predict(X_valid)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, sharey=True, figsize=(12, 4))
col=21
plt.subplot(131),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_naive[col, 0])
plt.subplot(132),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_DN[col, 0])
plt.subplot(133),plot_series(X_valid[col, :, 0], y_valid[col, 0], y_pred_DRNN[col, 0])

The predictions are good, but it is getting quite expensive (a lot more than the DN).\
To justify the cost, we have to look at more difficult tasks. 
We will now try to estimate 10 steps ahead. We could do this in two ways:
1) We predict one sample, put it in, predict the next.\
2) We predict all samples in the gap simultaneously.

For 1) we will use the same network we already trained. We just use it several times.
First we need to predict a time series for testing (because we need more steps ahead to test).

In [ ]:
np.random.seed(43) # not 42, as it would give the first series in the train set

series = generate_time_series(1, n_steps + 10)
X_new, Y_new = series[:, :n_steps], series[:, n_steps:]
X = X_new
for step_ahead in range(10):
    y_pred_one = model.predict(X[:, step_ahead:])[:, np.newaxis, :]
    X = np.concatenate([X, y_pred_one], axis=1)

Y_pred = X[:, n_steps:]

In [ ]:
print(X_new.shape,Y_new.shape,X.shape,Y_pred.shape)

In [ ]:
def plot_multiple_forecasts(X, Y, Y_pred):
    n_steps = X.shape[1]
    ahead = Y.shape[1]
    plot_series(X[0, :, 0])
    plt.plot(np.arange(n_steps, n_steps + ahead), Y[0, :, 0], "ro-", label="Actual")
    plt.plot(np.arange(n_steps, n_steps + ahead), Y_pred[0, :, 0], "bx-", label="Forecast", markersize=10)
    plt.axis([0, n_steps + ahead, -1, 1])
    plt.legend(fontsize=14)

plot_multiple_forecasts(X_new, Y_new, Y_pred)
plt.show()

Now we will do the prediction simultaneously. We need to generate the training dataset again with 10 steps ahead.

In [ ]:
#before
print(X_train.shape, X_valid.shape, X_test.shape)
print(y_train.shape, y_valid.shape, y_test.shape)

In [ ]:
np.random.seed(42)

n_steps = 50
series = generate_time_series(10000, n_steps + 10)
X_train, Y_train = series[:7000, :n_steps], series[:7000, -10:, 0]
X_valid, Y_valid = series[7000:9000, :n_steps], series[7000:9000, -10:, 0]
X_test, Y_test = series[9000:, :n_steps], series[9000:, -10:, 0]

In [ ]:
print(X_train.shape, X_valid.shape, X_test.shape)
print(y_train.shape, y_valid.shape, y_test.shape)

In [ ]:
X = X_valid
for step_ahead in range(10):
    y_pred_one = model.predict(X)[:, np.newaxis, :]
    X = np.concatenate([X, y_pred_one], axis=1)

Y_pred = X[:, n_steps:, 0]

In [ ]:
print(X_valid.shape,Y_valid.shape,Y_pred.shape)

In [ ]:
Y_naive_pred = Y_valid[:, -1:]
print('MSE for 10 step predictions naive approach',np.mean(keras.metrics.mean_squared_error(Y_valid, Y_valid[:,-1:])))
print('MSE for 10 step predictions of 10 values',np.mean(keras.metrics.mean_squared_error(Y_valid, Y_pred)))

Let us compare with the DN that works well before

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50, 1]),
    keras.layers.Dense(10)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

In [ ]:
Y_pred_DN10 = model.predict(X_valid)

In [ ]:
print(Y_pred.shape)
print(Y_pred_DN10.shape)

In [ ]:
Y_naive_pred = Y_valid[:, -1:]
print('MSE for 10 step predictions naive approach',np.mean(keras.metrics.mean_squared_error(Y_valid, Y_valid[:,-1:])))
print('MSE for 10 step predictions of 10 values',np.mean(keras.metrics.mean_squared_error(Y_valid, Y_pred_DN10)))

In [ ]:
def plot_multiple_forecasts(X, Y, Y_pred):
    print(X.shape, Y.shape, Y_pred.shape)
    n_steps = X.shape[1]    
    ahead = Y.shape[1]
    print(n_steps, ahead)
    plot_series(X[0, :, 0])
    plt.plot(np.arange(n_steps, n_steps + ahead), Y[0, :], "ro-", label="Actual")
    plt.plot(np.arange(n_steps, n_steps + ahead), Y_pred[0, :], "bx-", label="Forecast", markersize=10)
    plt.axis([0, n_steps + ahead, -1, 1])
    plt.legend(fontsize=14)
    
plot_multiple_forecasts(X_valid[:,:,:], Y_valid[:,:], Y_pred_DN10[:,:])
plt.show()

In [ ]:
plot_multiple_forecasts(X_valid[:,:,:], Y_valid[:,:], Y_pred[:,:])

Now let us do the DRNN again but all 10 steps at once, we need to put nsteps neurons in the last layer (like in DN).

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10)
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

In [ ]:
np.random.seed(43) # again we use a different number for random seed to prevent repeating the training series.
# Generate one time serie to predict with 50 steps and 10 steps ahead.
series = generate_time_series(1, 50 + 10)
X_new, Y_new = series[:, :50, :], series[:, -10:, :]
Y_pred = model.predict(X_new)[..., np.newaxis]

In [ ]:
plot_multiple_forecasts(X_new, Y_new, Y_pred)
plt.show()

In [ ]:
print('MSE for multistep DNN',model.evaluate(X_valid,Y_valid))

There is one more way to do multistep predictions. \
We can evaluate the 10 next steps for each step (instead of the last 10 steps only).\
These values can be used to estimate gradients during training but not for predictions since it is causal model.\
Causal means that the model can see the past not the future (like humans, not counting Nostradamus).

When generating the time series we need to allocate the targets in groups of 10 since the training now will be using 10 samples for each time steps. 

In [ ]:
np.random.seed(42)

n_steps = 50
series = generate_time_series(10000, n_steps + 10)
X_train = series[:7000, :n_steps]
X_valid = series[7000:9000, :n_steps]
X_test = series[9000:, :n_steps]
Y = np.empty((10000, n_steps, 10))
for step_ahead in range(1, 10 + 1):
    Y[..., step_ahead - 1] = series[..., step_ahead:step_ahead + n_steps, 0]
Y_train = Y[:7000]
Y_valid = Y[7000:9000]
Y_test = Y[9000:]

If we look at the shape of the target vector, we see it has an extra dimension of 10 for each of the time steps.

In [ ]:
print('y train size is (nbatches, nsteps, nsteps_ahead) =',Y_train.shape)

In [ ]:
print('X train is similar as before (nbatches, nsteps, 1) =',X_train.shape)

To predict 10 samples at the end of the series we just used the dense network with 10 neurons.\
Now, however, we need to predict 10 samples for each step. Keras has the TimeDistributed layer to help to do this.\
It needs to pass the Dense layer with 10 neurons (the same as before) but TimeDistributed will used it at each step.\
In addition, we need to use a custom metric that only calculates the error at the end (even when gradients are used for all time steps, we want the error to be measured at the end only).


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)
#Time distributed will call the dense layer at each step.
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])
#Customized metric to measure error only with the last 10 samples of each batch.
def last_time_step_mse(Y_true, Y_pred):
    return keras.metrics.mean_squared_error(Y_true[:, -1], Y_pred[:, -1])

model.compile(loss="mse", optimizer=keras.optimizers.Adam(lr=0.01), metrics=[last_time_step_mse])
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

Once again we create a testing time series (one batch, 50 steps + 10 steps ahead).\
We call X the series with the 50 steps only (without the 10), y is the series with the 10 steps ahead.\
The syntaxis in the prediction is a bit strange but it just does a reshaping to take the last 10 steps.\
Remember the TimeDistribute will predict 10 steps for each of the 50 steps, therefore making a matrix (50x10).\
We just want the 10 predictions for the last step.


In [ ]:
np.random.seed(43)

series = generate_time_series(1, 50 + 10)
X_new, Y_new = series[:, :50, :], series[:, 50:, :]
Y_pred_DRNN = model.predict(X_new)[:, -1][..., np.newaxis]
print(Y_new.shape,Y_pred_DRNN.shape)

In [ ]:
Y_pred0 = model.predict(X_new)
print(Y_pred0.shape)

In [ ]:
plt.plot(Y_pred_DRNN[0,:,0],label='reshaped')
plt.plot(Y_pred0[0, -1, :],'.',label='section at last row')
plt.legend()

In [ ]:
# we don't have values 50-60 in X, so let us plot 40:50
plt.plot(X_new[0,40:50,0],label='original')
plt.plot(Y_pred0[0,39,:],label='predicted')
plt.legend();plt.title('predictions in step 40')

In [ ]:
plot_multiple_forecasts(X_new, Y_new, Y_pred_DRNN)
plt.show()

In [ ]:
MSE_DRNN=model.evaluate(X_valid, Y_valid)
print(MSE_DRNN)

## Exercise 2: Long Short-Term Memory and Gate Recurrent Unit
Let us now use the LSTM, the GRU and compare with the DRNN.\
Everything is similar, except we use the LSTM layer instead of SimpleRNN.\
Also we need the TimeDistributed Layer to predict 10 samples at each time step.

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

Notice the time at each iteration was much smaller than before. LSTM can remember more with less effort.

In [ ]:
np.random.seed(43)

series = generate_time_series(1, 50 + 10)
X_new, Y_new = series[:, :50, :], series[:, 50:, :]
Y_pred_LSTM = model.predict(X_new)[:, -1][..., np.newaxis]
print(Y_new.shape,Y_pred.shape)

In [ ]:
plot_multiple_forecasts(X_new, Y_new, Y_pred_LSTM)
plt.show()

In [ ]:
MSE_LSTM=model.evaluate(X_valid, Y_valid)
print(MSE_LSTM)

Now the GRU

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

model = keras.models.Sequential([
    keras.layers.GRU(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))

This was half the time than the LSTM and much faster than DRNN

In [ ]:
Y_pred_GRU = model.predict(X_new)[:, -1][..., np.newaxis]
print(Y_new.shape,Y_pred_GRU.shape)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, sharey=True, figsize=(12, 4))
plt.subplot(131),plot_multiple_forecasts(X_new, Y_new, Y_pred)
plt.subplot(132),plot_multiple_forecasts(X_new, Y_new, Y_pred_LSTM)
plt.subplot(133),plot_multiple_forecasts(X_new, Y_new, Y_pred_GRU)

In [ ]:
MSE_GRU=model.evaluate(X_valid, Y_valid)
print(MSE_GRU)

In [ ]:
print(np.round(MSE_DRNN,4), np.round(MSE_LSTM,4), np.round(MSE_GRU,4))